# MDR-TB Treatment Outcome Prediction Model

This Google Colab notebook trains and evaluates the MDR-TB treatment outcome model for the academic prototype.

**Academic validity note:** the current project dataset is reconstructed from published aggregate statistics from Chanda (2024), not independently collected patient-level records. This notebook demonstrates a reproducible ML workflow and deployable artifact; it does not establish clinical validity.

## 1. Setup

Run from the repository root in Colab. If you upload the project ZIP, unzip it first and `cd` into the project folder.

In [2]:
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

'pip' is not recognized as an internal or external command,
operable program or batch file.


WindowsPath('//wsl.localhost/Ubuntu/home/collins/mdr-tb-treatment-outcomes-zambia-ml/notebooks')

## 2. Load Or Generate Dataset

The notebook loads `data/external/drtb_central_zambia_synthetic.csv` if available. Otherwise, it generates the project dataset from `data/external/circular_data_gen.py`.

In [3]:
import pandas as pd

csv_path = PROJECT_ROOT / "data" / "external" / "drtb_central_zambia_synthetic.csv"

if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    from data.external.circular_data_gen import generate_dataset
    df = generate_dataset()

df.head()

ModuleNotFoundError: No module named 'data'

## 3. Data Validity Statement

This dataset is suitable for demonstrating data science and software engineering workflow. It should not be presented as independent clinical data. The rows are reconstructed from published aggregate counts, so some inter-variable relationships are imposed by allocation and shuffling rather than observed directly.

In [ ]:
display(df.info())
display(df.describe(include="all").T.head(25))

## 4. Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.countplot(data=df, x="outcome", ax=axes[0, 0])
axes[0, 0].tick_params(axis="x", rotation=30)
axes[0, 0].set_title("Treatment outcomes")

sns.countplot(data=df, x="age_group", hue="outcome", ax=axes[0, 1])
axes[0, 1].tick_params(axis="x", rotation=30)
axes[0, 1].set_title("Outcome by age group")

sns.countplot(data=df, x="gender", hue="outcome", ax=axes[1, 0])
axes[1, 0].set_title("Outcome by gender")

top_districts = df["district"].value_counts().head(6).index
sns.countplot(data=df[df["district"].isin(top_districts)], x="district", hue="outcome", ax=axes[1, 1])
axes[1, 1].tick_params(axis="x", rotation=30)
axes[1, 1].set_title("Outcome by top districts")

plt.tight_layout()

## 5. Target Definition

The application predicts one of four classes: Treatment Success, Died, Lost to Follow Up, or Still on Treatment.

In [ ]:
from src.models.train import normalize_outcome, PREDICTION_FEATURES

model_df = df.copy()
model_df["outcome_class"] = model_df["outcome"].map(normalize_outcome)

X = model_df[PREDICTION_FEATURES]
y = model_df["outcome_class"]

display(y.value_counts())

## 6. Train/Test Split And Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from src.models.train import build_outcome_pipeline, RANDOM_STATE

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

model = build_outcome_pipeline()
model.fit(X_train, y_train)

model.classes_

## 7. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(xticks_rotation=30, cmap="Blues")
plt.title("Confusion matrix")
plt.show()

## 8. Feature Interpretation

For logistic regression, coefficients indicate how encoded feature values shift class scores. This supports model explainability in the prototype, but coefficients should not be interpreted as causal clinical evidence.

In [ ]:
preprocess = model.named_steps["preprocess"]
classifier = model.named_steps["classifier"]
feature_names = preprocess.get_feature_names_out()

coef_df = pd.DataFrame(classifier.coef_, index=classifier.classes_, columns=feature_names)
for class_name in classifier.classes_:
    print("\nTop positive features for", class_name)
    display(coef_df.loc[class_name].sort_values(ascending=False).head(8).to_frame("coefficient"))

## 9. Save Deployment Artifact

This cell trains through the repository training function and saves the artifact used by FastAPI and Streamlit.

In [ ]:
from src.models.train import train_outcome_model

result = train_outcome_model(df)
print("Model artifact:", result.artifact_path)
print("Metrics JSON:", result.metrics_path)
print("Model version:", result.model_version)
print("Held-out accuracy:", round(result.metrics["accuracy"], 3))

## 10. Academic Limitations And Future Work

- The dataset is reconstructed from aggregate counts, not independent patient-level clinical records.
- Performance metrics demonstrate software/model workflow, not clinical validity.
- Future work should use approved de-identified patient-level data, external validation, calibration analysis, subgroup fairness checks, and clinical governance review.
- Candidate models such as random forest, gradient boosting, and calibrated logistic regression can be compared after the data validity problem is addressed.